## Introduction

In this tutorial, we will build a character-level text autocomplete model using a Recurrent Neural Network (RNN) in PyTorch. We will train the model on the text from "warandpeace.txt". This project will help you understand how RNNs can be implemented for text generation tasks and their application in building your own autocomplete model.


## Importing Necessary Libraries

In [48]:
# This is Cell #1

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import random
import re


## Setting Up the Device

In [49]:
# This is Cell #2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Reading and Preprocessing the Data

Now it is time to prepare our training data.


In [50]:
# This is Cell #3

def read_file(filename):
    with open(filename, "r", encoding="utf-8") as file:
        text = file.read().lower()
        # Keep only lowercase letters and standard punctuation (.,!?;:()[])
        text = re.sub(r'[^a-z.,!?;:()\[\] ]+', '', text)
    return text

sequence = read_file("warandpeace.txt")


### Here we will train our model with a simple sequence

We will start by training our model with a simple sequence and repettitive sequence such as `"abcdefghijklmnopqrstuvwxyzabcdef..."`, and we will see if our RNN is capable of learning that pattern or not. This will help you easily verify if your RNN is working correctly or not.

In [39]:
# This is Cell #4

sequence = "abcdefghijklmnopqrstuvwxyz" * 100

## Create Character Mappings

Creating character mappings is essential because RNNs require numerical input to process data. By mapping each unique character to an index and creating a reverse mapping, we convert text data into numerical sequences that the model can understand. This step allows us to encode input text for training and decode the model's output back into readable characters during text generation.



In [51]:
# This is Cell #5

#TODO: Create a list of unique characters from the text sequence
vocab = sorted(set(sequence)) 

#TODO: Create two dictionaries for character-index mappings that map each character in vocab to a unique index and vice versa
char_to_idx = {char: idx for idx, char in enumerate(vocab)} 
idx_to_char = {idx: char for idx, char in enumerate(vocab)} 

#TODO: Convert the entire text based data into numerical data
data = [char_to_idx[char] for char in sequence]

## Defining the CharDataset Class

Now we will create a custom dataset class to generate sequences and targets for training

Creating a custom `CharDataset` class is crucial because it prepares our text data into input sequences and target sequences that the RNN can learn from. By organizing the data this way, we can efficiently feed batches of sequences into the model during training, allowing it to learn the patterns of character sequences in the text.

In [52]:
# This is Cell #6

class CharDataset(Dataset):
    def __init__(self, data, sequence_length, stride, vocab_size):
        self.data = data
        self.sequence_length = sequence_length
        self.stride = stride
        self.vocab_size = vocab_size
        self.sequences = []
        self.targets = []
        
        # Create overlapping sequences with stride
        for i in range(0, len(data) - sequence_length, stride):
            self.sequences.append(data[i:i + sequence_length])
            self.targets.append(data[i + 1:i + sequence_length + 1])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return sequence, target
    

## Setting Hyperparameters

Now we will set our model's hyperparameters for our training process

Setting hyperparameters is important because they define the model's architecture and training behavior. They determine how the RNN processes data, learns patterns, and how quickly it converges during training. Properly chosen hyperparameters can significantly improve model performance and is a key step in training of models

Set the following hyperparameters for your model in the code cell below:
`sequence_length`, `stride`, `embedding_dim`, `hidden_size`, `num_layers`, `learning_rate`, `num_epochs`, `batch_size`, `vocab_size`.

In [53]:
# This is Cell #7

#TODO: Set your model's hyperparameters

sequence_length = 50  # Length of each input sequence
stride = 1            # Stride for creating sequences
embedding_dim = 64     # Dimension of character embeddings
hidden_size = 128      # Number of features in the hidden state of the RNN
learning_rate = 0.001  # Learning rate for the optimizer
num_epochs = 1         # Number of epochs to train
batch_size = 16       # Batch size for training
vocab_size = len(vocab)
input_size = len(vocab)
output_size = len(vocab)
num_layers = 2

After you have set your hyperparameters in the code cell above, very breifly tell what is the role of each of the hyperparameter that you have defined above.

TODO: Explain below
> Here
sequence_length: This defines the length of each input sequence that the model will process.

> stride: The stride determines the step size when moving through the data to create sequences. A stride of 1 means that the sequences will overlap by sequence_length - 1 characters, providing the model with many training samples that capture the character-to-character transitions.

> embedding_dim: Defines the size of the vector space for character embeddings. Higher dimensions capture richer relationships between characters.

> hidden_size: The number of features in the hidden state of the RNN. Larger values allow the model to learn more complex patterns.

> num_layers: Specifies the number of stacked RNN layers. More layers increase the model's capacity to learn hierarchical representations.

> learning_rate: Controls the step size of the optimizer during weight updates. Smaller values ensure stable convergence, while larger values speed up training but risk overshooting the minimum.

> num_epochs: The number of complete passes through the training dataset. More epochs allow the model to learn better but can lead to overfitting.

> batch_size: The number of sequences processed together in one forward/backward pass. Larger batch sizes make training faster but require more memory.

> vocab_size: Represents the total number of unique characters in the dataset. It's used to define the input and output dimensions of the model.

> input_size: Specifies the size of the input layer, which is equal to the vocabulary size.

> output_size: Sets the size of the output layer of the model.

## Splitting Data into Training and Testing Sets

By now at this point in class, I'm confident that you know why we do this, so I'm not gonna say a lot here, let's jump right into the todo.

In [54]:
# This is Cell #8

data_tensor = torch.tensor(data, dtype=torch.long)

#TODO: Convert the data into a pytorch tensor and split the data into 90:10 ratio
train_size = int(0.9 * len(data_tensor)) 
train_data = data_tensor[:train_size]  
test_data = data_tensor[train_size:]   


## Creating Data Loaders

Now we will create data loaders for easy batching during training and testing.

Creating data loaders is essential to batch the data during training and testing. Batching allows the RNN to process multiple sequences in parallel, which speeds up training and makes better use of computational resources. 
We will also use Data loaders to shuffle the batched data, which is important for training models that generalize well.

Make sure to set `drop_last=True`

In [55]:
# This is Cell #9
from torch.utils.data import DataLoader

train_dataset = CharDataset(train_data, sequence_length, stride, vocab_size)
test_dataset = CharDataset(test_data, sequence_length, stride, vocab_size)

#TODO: Initialize the training and testing data loader with batching and shuffling equal to True for training (and shuffling = False for testing)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)  
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)  

total_batches = len(train_loader)


## Defining the RNN Model

Here we will define our character-level RNN model.

In [56]:
# This is Cell #10

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, embedding_dim=30):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = torch.nn.Embedding(output_size, embedding_dim)
        self.W_e = nn.Parameter(torch.randn(hidden_size, embedding_dim) * 0.01)  # Smaller std
        self.b_e = nn.Parameter(torch.zeros(hidden_size))
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)  # Smaller std
        self.b_h = nn.Parameter(torch.zeros(hidden_size)) 
        #TODO: set the fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        """
        x in [b, l] # b is batch_size and l is sequence length
        """
        x_embed = self.embedding(x)  # [b=batch_size, l=sequence_length, e=embedding_dim]
        b, l, _ = x_embed.size()
        x_embed = x_embed.transpose(0, 1) # [l, b, e]
        if hidden is None:
            h_t_minus_1 = self.init_hidden(b)
        else:
            h_t_minus_1 = hidden
        output = []
        for t in range(l):
            # RNN equation from the lecture 
            # We add a bias as well to expand the range of learnable functions
            h_t = torch.tanh(x_embed[t] @ self.W_e.T + self.b_e + h_t_minus_1 @ self.W_h.T + self.b_h) # [b, e]
            output.append(h_t)
            h_t_minus_1 = h_t
        output = torch.stack(output) # [l, b, e]
        output = output.transpose(0, 1) # [b, l, e]
        final_hidden = h_t.clone() # [b, h]
        logits = self.fc(output) # [b, l, vocab_size=v] 
        return logits, final_hidden
    
    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.hidden_size).to(device)


For a basic high level understanding of what is the CharRNN model that you just defined above, it consists of an embedding layer, an RNN layer, and a fully connected layer. Then embedding layer converts character indices into embeddings. Then RNN processes the embeddings and captures sequential information. Then finally the fully connected layer maps the RNN outputs to the vocabulary size for character prediction.


# Initializing the Model, Loss Function, and Optimizer

Now we will create an instance of the model that we just defined above and set up the loss function and optimizer. Then we will define a loss function, that evaluates the model's prediction against the true targets, and attaches a cost (number) on how good/bad the model is doing. During our training process, it is this cost that we try to minimize by tweaking the weights of the network. 

Then we will set up an optimizer, which will update the model's parameters based on the loss returned by the our loss function. This is how our model will learn over time.


In [57]:
# This is Cell #12

#TODO: Initialize your RNN model
model = CharRNN(input_size=input_size, hidden_size=hidden_size, output_size=output_size, embedding_dim=embedding_dim).to(device)

#TODO: Define the loss function (use cross entropy loss)
criterion = nn.CrossEntropyLoss()

#TODO: Initialize your optimizer passing your model parameters and training hyperparameters
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


## Training the Model

Now finally, after all the setup that we have done, we can train our RNN. 

A basic idea high level idea of what we will do here is we will loop over epochs and batches to train the model. 
We will Initialize the hidden state at the beginning of each epoch. For each batch, we will reset the gradients, perform a forward pass, compute the loss, perform backpropagation, and update the model parameters. Then we detach the hidden state to prevent gradients from backpropagating through previous batches. We ill repeat this process for each batch. And finally we will calculate the average loss and accuracy for each epoch.
By performing forward and backward passes, calculating loss, and updating the model parameters, we enable the RNN to improve its predictions with each epoch.

In [58]:
# This is Cell #13

for epoch in range(num_epochs):
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(train_loader), total=total_batches, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        with torch.no_grad():
            # Calculate accuracy
            _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

            correct_predictions += (predicted_indices == batch_targets).sum().item()
            total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

Epoch 1/1:   0%|                                                                             | 0/175494 [00:00<?, ?it/s]/var/folders/rw/dqhtql8d5p92lf537q7kks2r0000gn/T/ipykernel_37918/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/var/folders/rw/dqhtql8d5p92lf537q7kks2r0000gn/T/ipykernel_37918/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Epoch 1/1: 100%|███████████████████████████████████████████████████████████████| 175494/175494 [27:01<00:00, 108.26it/s]

Epoch [1/1], Loss: 1.5906, Accuracy: 53.33%


## Check your loss

The training loss of your model when trained with a simple sequence like `"abcdefghijklmnopqrstuvwxyz" * 100` should be extremely close to zero. If that's not the case, go back and fix your bugs ;)

If you have acheived a training loss of 0 or extremley close to 0, then congratulations, lets move on to train your model with a bit more complicated sequence. That is our old favorite book, `warandpeace.txt`.

### Read the `warandpeace.txt` file

In [59]:
# This is Cell #14

sequence = read_file('warandpeace.txt')

### Now Follow the instructions

1. Re-run Cell #5 to re-create character mappings for `warandpeace.txt`
2. Re-run Cell #7 to re-initialize hyperparameters
3. Re-run Cell #8 to split and create training and testing data with `warandpeace.txt` as your corpus
4. Re-run Cell #9 to set up data loaders with `warandpeace.txt` data
5. Re-run Cell #12 to re-initialize a new model object (maybe ask yourself why can't you use the previous model that was trained on the simple `"abc..."` corpus)
6. Re-run Cell #13 to train the new model with `warandpeace.txt` data.
   

## Evaluating the Model

After training, we evaluate the model on the test data.

In [62]:
# This is Cell #15

with torch.no_grad():
    #TODO: Write the testing loop for your trained model by refering to the training loop code given to you above
    total_test_loss, correct_test_predictions, total_test_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in enumerate(test_loader):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()
   
        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))

        _, predicted_indices = torch.max(output, dim=2)  
        correct_test_predictions += (predicted_indices == batch_targets).sum().item()
        total_test_predictions += batch_targets.numel()  

        total_test_loss += loss.item()

    avg_loss = total_test_loss / len(test_loader)
    accuracy = (correct_test_predictions / total_test_predictions) * 100  # Convert to percentage


    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

/var/folders/rw/dqhtql8d5p92lf537q7kks2r0000gn/T/ipykernel_37918/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/var/folders/rw/dqhtql8d5p92lf537q7kks2r0000gn/T/ipykernel_37918/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)


Test Loss: 1.6327, Test Accuracy: 52.01%


## Generating Text with the Trained Model

In this part of the assignment, your task is to implement the `generate_text` function, which uses a trained RNN model to generate text character-by-character, continuing from a given input. The function will produce an extended sequence by repeatedly predicting and appending the next character to the input.

### What the function is supposed to do?

1. Take an initial input text of length `n` from the user, convert it into indices using a predefined vocabulary (char_to_idx).
2. Use a trained model to predict the next character in the sequence.
3. Append the predicted character to the input, extend the input sequence, and repeat the process until `k` additional characters are generated.
4. Return the generated text, including the original input and the newly predicted characters.


In [ ]:
#Test
# This is Cell #16

def sample_from_output(logits, temperature=1.0):
    """
    Sample from the logits with temperature scaling.
    logits: Tensor of shape [batch_size, vocab_size] (raw scores, before softmax)
    temperature: a float controlling the randomness (higher = more random)
    """
    # Apply temperature scaling to logits (increase randomness with higher values)
    scaled_logits = logits / temperature  # Scale the logits by temperature
    # Apply softmax to convert logits to probabilities
    probabilities = F.softmax(scaled_logits, dim=1)
    
    # Sample from the probability distribution
    sampled_idx = torch.multinomial(probabilities, 1)  # Sample one index from the probability distribution
    return sampled_idx

def generate_text(model, start_text, n, k, temperature=1.0):
    """
        model: The trained RNN model used for character prediction.
        start_text: The initial string of length `n` provided by the user to start the generation.
        n: The length of the initial input sequence.
        k: The number of additional characters to generate.
        temperature: A scaling factor for randomness in predictions.
    """
    model.eval() 
    start_text = start_text.lower()
    input_indices = [char_to_idx.get(char, char_to_idx[' ']) for char in start_text]
    input_indices_tensor = torch.tensor(input_indices, dtype=torch.long).unsqueeze(0).to(device)  # Shape [1, n]

    hidden = model.init_hidden(1) 

    with torch.no_grad():
        output, hidden = model(input_indices_tensor, hidden)

    generated_indices = input_indices.copy()

    last_char_index = input_indices[-1]

    for _ in range(k):
        input_tensor = torch.tensor([[last_char_index]], dtype=torch.long).to(device)  
        with torch.no_grad():
            output, hidden = model(input_tensor, hidden)
            logits = output[:, -1, :]  
            next_char_index = sample_from_output(logits, temperature=temperature)
            next_char_index = next_char_index.item()
            generated_indices.append(next_char_index)
            last_char_index = next_char_index
    generated_text = ''.join([idx_to_char[idx] for idx in generated_indices])

    return generated_text

print("Training complete. Now you can generate text.")
while True:
    start_text = input("Enter the initial text (n characters, or 'exit' to quit): ")
    
    if start_text.lower() == 'exit':
        print("Exiting...")
        break
    
    n = len(start_text) 
    k = int(input("Enter the number of characters to generate: "))
    temperature_input = input("Enter the temperature value (1.0 is default, >1 is more random): ")
    temperature = float(temperature_input) if temperature_input else 1.0
    
    completed_text = generate_text(model, start_text, n, k, temperature)
    
    print(f"Generated text: {completed_text}")


Training complete. Now you can generate text.


Enter the initial text (n characters, or 'exit' to quit):  and
Enter the number of characters to generate:  100
Enter the temperature value (1.0 is default, >1 is more random):  1.2


Generated text: and matherentwential e med through his man for themselfto the eatblus slvreed, daskedies.bors.count and


Enter the initial text (n characters, or 'exit' to quit):  and
Enter the number of characters to generate:  100
Enter the temperature value (1.0 is default, >1 is more random):  1


Generated text: and sat that miving a honevely  t something god, cade cours question.both evidently whier that all thei


...

Generated text: and have standing the sound and the sound and the sound and the sound and the sound and the sound and t


Enter the initial text (n characters, or 'exit' to quit):  and
Enter the number of characters to generate:  50
Enter the temperature value (1.0 is default, >1 is more random):  0.3


Generated text: andenlial did not all was a service of the sound and 


Enter the initial text (n characters, or 'exit' to quit):  and
Enter the number of characters to generate:  20
Enter the temperature value (1.0 is default, >1 is more random):  0.25


Generated text: and still and the fact 


## Report section

In your report, describe your experiments and observations when training the model with two datasets: (1) the sequence `"abcdefghijklmnopqrstuvwxyz" * 100` and (2) the text from `warandpeace.txt`. Include the final loss values for both datasets and discuss how the generated text differed between the two. Explain the impact of changing the `temperature` parameter on the text generation, and provide examples. Reflect on the challenges you faced, your thought process during implementation, and the key insights you gained about RNNs and sequence modeling.

1. `abcdefghijklmnopqrstuvwxyz`
* With the intial hyperparamters given to us, I got really bad results with an accuracy % of around 1.92%.
* I added num_layers = 2, which got the accuracy % of 3.85%.
* I changed the learning rate to 0.01, the sequence_length = 50, the embedding_dim = 64, and hidden_size = 128. This gave an accuracy % of 67%.
* I made the stride = 1, which gave an accuracy score of 96.55%.
* I changed the batch_size to 16 whcih then gave an accuracy score of 99.18%.

2. `warandpeace.txt`
* With these hyperparameters above I got less than 50% accuracy.
* I changed the learning rate to 0.001 and that resulted in better accuracy.

3. `Impact of changing temperature`
* I tested this with some simple examples. The inital text was `and`, and the number of characters to generate I tested on was 20,40,50,100. For all of these results I have seen that the lower the temperature the better the results (until we reach the extremes). Initially I thought that since the default temperature was 1, that might give the best results, but that was not the case. However, I also noticed that when I made the temperature very low, the model would repeat words.
Ex: Constants: Initial text: and, number of characters to generate: 50
1. temperature = 1.2 --> Generated text: `and that tugning yearside faceful petermentsowntcount`
2. temperature = 1.0 --> Generated text: `andushing rewards bonexemrehapartant, and said poware`
3. temperature = 0.7 --> Generated text: `andens was to come to be trying the field and a somet`
4. temperature = 0.5 --> Generated text: `andloggself in the came a fire, and she told the enem`
5. temperature = 0.25 --> Generated text: `and still and the fact`
6. temperature = 0.1 --> Generated text: `and the countess was a countess was a countess was a`
7. temperature = 0.001 --> Generated text: `and the countess was a countess was a countess was a `

4. `Challenges`
* The main challenges that I faced this assignment was fine-tuning the hyperparamaters so that they would result in good final results. It took some time to figure out what the 'optimal' parameters were. I researched on the internet and even asked gpt383 for help, and after experimenting for quite a bit of time I finally got some good results.

5. `Key Insights`
* Fine-tuning hyperparameters like learning rate, sequence length, and batch size is critical for optimizing RNN performance.
* Adjusting temperature during text generation are crucial to achieve good results, but extreme values lead to repetition or randomness.
* 0.25 seemed to be the most optimal temperature for the best results.